In [2]:
import pandas as pd
import plotly.express as px


df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')


df.columns = df.columns.str.replace(' ', '_').str.lower()


df['order_date'] = pd.to_datetime(df['order_date'])
df['month_year'] = df['order_date'].dt.to_period('M').astype(str)


df['sales'] = df['sales'].round(2)
df['profit'] = df['profit'].round(2)


df['profit_margin'] = (df['profit'] / df['sales'] * 100).round(2)

print("Data Cleaned. Ready for Analysis.")
print(df)

Data Cleaned. Ready for Analysis.
      row_id        order_id order_date   ship_date       ship_mode  \
0          1  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
1          2  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
2          3  CA-2016-138688 2016-06-12   6/16/2016    Second Class   
3          4  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
4          5  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
...      ...             ...        ...         ...             ...   
9989    9990  CA-2014-110422 2014-01-21   1/23/2014    Second Class   
9990    9991  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9991    9992  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9992    9993  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9993    9994  CA-2017-119914 2017-05-04    5/9/2017    Second Class   

     customer_id     customer_name    segment        country             city  \
0       CG-12520       Claire Gu

In [3]:

trend = df.groupby('month_year')['sales'].sum().reset_index()


top_products = df.groupby('product_name')['sales'].sum().sort_values(ascending=False).head(5)


region_perf = df.groupby('region')[['sales', 'profit']].sum().reset_index()
print(df)

      row_id        order_id order_date   ship_date       ship_mode  \
0          1  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
1          2  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
2          3  CA-2016-138688 2016-06-12   6/16/2016    Second Class   
3          4  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
4          5  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
...      ...             ...        ...         ...             ...   
9989    9990  CA-2014-110422 2014-01-21   1/23/2014    Second Class   
9990    9991  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9991    9992  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9992    9993  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9993    9994  CA-2017-119914 2017-05-04    5/9/2017    Second Class   

     customer_id     customer_name    segment        country             city  \
0       CG-12520       Claire Gute   Consumer  United States      

In [4]:

losses = df[df['profit'] < 0].groupby('sub-category')['profit'].sum().sort_values()
print("Top Loss-Making Sub-Categories:")
print(losses.head(3))
print(df)

Top Loss-Making Sub-Categories:
sub-category
Binders    -38510.57
Tables     -32412.21
Machines   -30118.71
Name: profit, dtype: float64
      row_id        order_id order_date   ship_date       ship_mode  \
0          1  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
1          2  CA-2016-152156 2016-11-08  11/11/2016    Second Class   
2          3  CA-2016-138688 2016-06-12   6/16/2016    Second Class   
3          4  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
4          5  US-2015-108966 2015-10-11  10/18/2015  Standard Class   
...      ...             ...        ...         ...             ...   
9989    9990  CA-2014-110422 2014-01-21   1/23/2014    Second Class   
9990    9991  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9991    9992  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9992    9993  CA-2017-121258 2017-02-26    3/3/2017  Standard Class   
9993    9994  CA-2017-119914 2017-05-04    5/9/2017    Second Class   

     custo

In [5]:

regional_analysis = df.groupby('region').agg({
    'sales': 'sum',
    'profit': 'sum',
    'profit_margin': 'mean'
}).sort_values(by='profit', ascending=False)

print("Regional Efficiency Report:")
print(regional_analysis)

Regional Efficiency Report:
             sales     profit  profit_margin
region                                      
West     725457.76  108418.31      21.948305
East     678781.30   91522.50      16.722051
South    391721.83   46749.49      16.351111
Central  501239.76   39706.24     -10.409475


In [6]:

df.to_csv('Superstore_Final_Analysis.csv', index=False)


from google.colab import files
files.download('Superstore_Final_Analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


total_sales = f"${df['sales'].sum():,.0f}"
total_profit = f"${df['profit'].sum():,.0f}"
avg_margin = f"{(df['profit'].sum() / df['sales'].sum() * 100):.1f}%"


fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Sales Trend Over Time", "Profit by Region",
                    "Sales by Category", "Top 5 Products by Sales"),
    vertical_spacing=0.12,
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "pie"}, {"type": "bar"}]]
)


fig.add_trace(
    go.Scatter(x=trend['month_year'], y=trend['sales'], name="Sales", line=dict(color='#3366CC', width=3)),
    row=1, col=1
)


fig.add_trace(
    go.Bar(x=region_perf['region'], y=region_perf['profit'], name="Profit", marker_color='#109618'),
    row=1, col=2
)


cat_data = df.groupby('category')['sales'].sum().reset_index()
fig.add_trace(
    go.Pie(labels=cat_data['category'], values=cat_data['sales'], hole=.4, name="Category"),
    row=2, col=1
)


fig.add_trace(
    go.Bar(x=top_products.values, y=top_products.index, orientation='h', name="Top Products", marker_color='#FF9900'),
    row=2, col=2
)


fig.update_layout(
    title_text=f"Superstore Executive Summary Dashboard <br><sup>Total Sales: {total_sales} | Total Profit: {total_profit} | Margin: {avg_margin}</sup>",
    height=900,
    template="plotly_white",
    showlegend=False
)

fig.show()

In [11]:
fig.write_html("Superstore_Dashboard.html")